In [0]:
%python
# Configuración general del laboratorio
 
import requests
import time
import json
from datetime import datetime, date, UTC
import pandas as pd
 
from pyspark.sql import functions as F
from pyspark.sql import Row
 
CATALOG = "workspace"
SCHEMA = "bigdata"
VOLUME = "lab_files"
 
TABLE_NAME = "coin_prices_lab2"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
 
COINS = ["bitcoin", "ethereum", "solana","dogcoin"]
VS_CURRENCY = "usd"
DAYS = 60
 
# Timestamp UTC con zona horaria
RUN_TS = datetime.now(UTC) #centrada en cero
 
# String segura para rutas
RUN_DATE = RUN_TS.strftime("%Y-%m-%d") 
 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
 
RAW_BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lab2/raw/{RUN_DATE}"
PROCESSED_BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lab2/processed/{RUN_DATE}"
 
CSV_OUTPUT_PATH = f"{PROCESSED_BASE_PATH}/csv"
PARQUET_OUTPUT_PATH = f"{PROCESSED_BASE_PATH}/parquet"
 
print("Tabla destino:", FULL_TABLE_NAME)
print("Fecha de ejecución:", RUN_DATE)
print("RAW_BASE_PATH:", RAW_BASE_PATH)
print("PROCESSED_BASE_PATH:", PROCESSED_BASE_PATH)
print(f"Esquema listo: {CATALOG}.{SCHEMA}")
print(f"Volumen listo: {CATALOG}.{SCHEMA}.{VOLUME}")

Tabla destino: workspace.bigdata.coin_prices_lab2
Fecha de ejecución: 2026-08-29
RAW_BASE_PATH: /Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29
PROCESSED_BASE_PATH: /Volumes/workspace/bigdata/lab_files/lab2/processed/2026-08-29
Esquema listo: workspace.bigdata
Volumen listo: workspace.bigdata.lab_files


In [0]:
%python
def fetch_market_chart(
    coin: str,
    vs_currency: str = "usd",
    days: int = 30,
    max_retries: int = 10
) -> dict:
    """Consulta precios históricos desde CoinGecko con reintentos básicos."""
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {
        "vs_currency": vs_currency,
        "days": days
    }
 
    response = None
 
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=30)
 
        if response.status_code == 200:
            return response.json()
 
        if response.status_code in (429, 500, 502, 503, 504):
            wait_time = 2 * (attempt + 1)
            print(f"Reintento {attempt + 1}/{max_retries} para {coin}. Esperando {wait_time}s...")
            time.sleep(wait_time)
        else:
            response.raise_for_status()

In [0]:
%python
all_processed = []
 
for coin in COINS:
    print(f"Consultando datos para: {coin}")
    data = fetch_market_chart(coin=coin, vs_currency=VS_CURRENCY, days=DAYS)
 
    raw_path = f"{RAW_BASE_PATH}/{coin}.json"
    dbutils.fs.put(raw_path, json.dumps(data), overwrite=True)
    print(f"RAW guardado en: {raw_path}")
 
    prices = data.get("prices", []) #DATOS JSON O SEMIESTRUCTURADOS Y SE PONEN EN DICCIONARIOS Y EN DATAFRAME SOLO SE QUEDAN COMO LA LLAVE PRICES
  
 
    if not prices:
        print(f"No se encontraron precios para {coin}")
        continue
 
    pdf = pd.DataFrame(prices, columns=["timestamp_ms", "price"]) # SE PASA A DF - CAMBIO A FORMATO 
    pdf["timestamp"] = pd.to_datetime(pdf["timestamp_ms"], unit="ms", utc=True)
    pdf["date"] = pdf["timestamp"].dt.date
    pdf["coin"] = coin
    pdf["vs_currency"] = VS_CURRENCY
 
    pdf = pdf[["coin", "vs_currency", "timestamp", "date", "price"]]
    all_processed.append(pdf)
 
print("Extracción completada.")

Consultando datos para: bitcoin
Wrote 157725 bytes.
RAW guardado en: /Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29/bitcoin.json
Consultando datos para: ethereum
Reintento 1/10 para ethereum. Esperando 2s...
Reintento 2/10 para ethereum. Esperando 4s...
Reintento 3/10 para ethereum. Esperando 6s...
Reintento 4/10 para ethereum. Esperando 8s...
Reintento 5/10 para ethereum. Esperando 10s...
Reintento 6/10 para ethereum. Esperando 12s...
Reintento 7/10 para ethereum. Esperando 14s...
Reintento 8/10 para ethereum. Esperando 16s...
Wrote 157982 bytes.
RAW guardado en: /Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29/ethereum.json
Consultando datos para: solana
Wrote 156392 bytes.
RAW guardado en: /Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29/solana.json
Consultando datos para: dogcoin
Reintento 1/10 para dogcoin. Esperando 2s...
Reintento 2/10 para dogcoin. Esperando 4s...
Reintento 3/10 para dogcoin. Esperando 6s...
Reintento 4/10 para dogcoin. Esperando 8s...

---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File <command-7539788705127110>, line 5
      3 for coin in COINS:
      4     print(f"Consultando datos para: {coin}")
----> 5     data = fetch_market_chart(coin=coin, vs_currency=VS_CURRENCY, days=DAYS)
      7     raw_path = f"{RAW_BASE_PATH}/{coin}.json"
      8     dbutils.fs.put(raw_path, json.dumps(data), overwrite=True)

File <command-6348549791020132>, line 27, in fetch_market_chart(coin, vs_currency, days, max_retries)
     25     time.sleep(wait_time)
     26 else:
---> 27     response.raise_for_status()

File /databricks/python/lib/python3.12/site-packages/requests/models.py:1024, in Response.raise_for_status(self)
   1019     http_error_msg = (
   1020         f"{self.status_code} Server Error: {reason} for url: {self.url}"
   1021     )
   1023 if http_error_msg:
-> 1024     raise HTTPError(http_error_msg, response=self)



In [0]:
%python
print("Archivos RAW generados:")
display(dbutils.fs.ls(RAW_BASE_PATH))

Archivos RAW generados:


path,name,size,modificationTime
dbfs:/Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29/bitcoin.json,bitcoin.json,157725,1788019457000
dbfs:/Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29/ethereum.json,ethereum.json,157982,1788019530000
dbfs:/Volumes/workspace/bigdata/lab_files/lab2/raw/2026-08-29/solana.json,solana.json,156392,1788019530000


In [0]:
%python
df_pd = pd.concat(all_processed, ignore_index=True) #TABULAR
df_pd.head()

,coin,vs_currency,timestamp,date,price
0,bitcoin,usd,2026-06-30 16:00:00+00:00,2026-06-30,58297.401144
1,bitcoin,usd,2026-06-30 17:00:00+00:00,2026-06-30,58410.110150
2,bitcoin,usd,2026-06-30 18:00:00+00:00,2026-06-30,58348.743085
3,bitcoin,usd,2026-06-30 19:00:00+00:00,2026-06-30,58590.967113
4,bitcoin,usd,2026-06-30 20:00:00+00:00,2026-06-30,58726.609031


In [0]:
%python
df_spark = spark.createDataFrame(df_pd) #TABULAR
 
df_spark = (
    df_spark
    .withColumn("timestamp", F.col("timestamp").cast("timestamp"))
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("price", F.col("price").cast("double"))
)
 
display(df_spark)

coin,vs_currency,timestamp,date,price
bitcoin,usd,2026-06-30T16:00:00.000Z,2026-06-30,58297.401144206626
bitcoin,usd,2026-06-30T17:00:00.000Z,2026-06-30,58410.110150264336
bitcoin,usd,2026-06-30T18:00:00.000Z,2026-06-30,58348.74308469594
bitcoin,usd,2026-06-30T19:00:00.000Z,2026-06-30,58590.967113077684
bitcoin,usd,2026-06-30T20:00:00.000Z,2026-06-30,58726.609030518965
bitcoin,usd,2026-06-30T21:00:00.000Z,2026-06-30,58657.09845015682
bitcoin,usd,2026-06-30T22:00:00.000Z,2026-06-30,58559.8204976322
bitcoin,usd,2026-06-30T23:00:00.000Z,2026-06-30,58531.71165559294
bitcoin,usd,2026-07-01T00:00:00.000Z,2026-07-01,58566.091991843154
bitcoin,usd,2026-07-01T01:00:00.000Z,2026-07-01,58488.330643900146


In [0]:
%python 
df_spark.printSchema()
print(f"Total de registros: {df_spark.count()}")
print(f"Total de monedas: {df_spark.select('coin').distinct().count()}")

root
 |-- coin: string (nullable = true)
 |-- vs_currency: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)

Total de registros: 4322
Total de monedas: 3


In [0]:
%python
#CSV PARQUET
(
    df_spark.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(CSV_OUTPUT_PATH)
)
 
print(f"CSV guardado en: {CSV_OUTPUT_PATH}")


CSV guardado en: /Volumes/workspace/bigdata/lab_files/lab2/processed/2026-08-29/csv


In [0]:
%python
#parquet
(
    df_spark.write
    .mode("overwrite")
    .parquet(PARQUET_OUTPUT_PATH)
)
 
print(f"Parquet guardado en: {PARQUET_OUTPUT_PATH}")

Parquet guardado en: /Volumes/workspace/bigdata/lab_files/lab2/processed/2026-08-29/parquet


In [0]:
%python
#comparacion de formatos

df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_OUTPUT_PATH)
)
 
df_parquet = spark.read.parquet(PARQUET_OUTPUT_PATH)
 
print("Registros en CSV:", df_csv.count())
print("Registros en Parquet:", df_parquet.count())
 
print("Esquema CSV")
df_csv.printSchema()
 
print("Esquema Parquet")
df_parquet.printSchema()

Registros en CSV: 4322
Registros en Parquet: 4322
Esquema CSV
root
 |-- coin: string (nullable = true)
 |-- vs_currency: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)

Esquema Parquet
root
 |-- coin: string (nullable = true)
 |-- vs_currency: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)



In [0]:
%python
#tablas delta

spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
 
(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FULL_TABLE_NAME)
)
 
print(f"Tabla Delta creada: {FULL_TABLE_NAME}")

Tabla Delta creada: workspace.bigdata.coin_prices_lab2


In [0]:
%python
#comparacion de tamaños

def get_directory_size_bytes(path: str) -> int:
    """Suma recursivamente el tamaño de todos los archivos dentro de una ruta."""
    total_size = 0
 
    for item in dbutils.fs.ls(path):
        if item.isDir():
            total_size += get_directory_size_bytes(item.path)
        else:
            total_size += item.size
 
    return total_size
 
 
def bytes_to_human_readable(num_bytes: int) -> str:
    """Convierte bytes a una unidad legible."""
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
 
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
 
 
json_size = get_directory_size_bytes(RAW_BASE_PATH)
csv_size = get_directory_size_bytes(CSV_OUTPUT_PATH)
parquet_size = get_directory_size_bytes(PARQUET_OUTPUT_PATH)
DELTA_COMPARE_PATH = f"{PROCESSED_BASE_PATH}/delta_compare"
(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .save(DELTA_COMPARE_PATH)
)
delta_size = get_directory_size_bytes(DELTA_COMPARE_PATH)
 
print("Tamaño Delta:", bytes_to_human_readable(delta_size))
print("Tamaño JSON RAW:", bytes_to_human_readable(json_size))
print("Tamaño CSV:", bytes_to_human_readable(csv_size))
print("Tamaño Parquet:", bytes_to_human_readable(parquet_size))

Tamaño Delta: 72.47 KB
Tamaño JSON RAW: 461.03 KB
Tamaño CSV: 279.79 KB
Tamaño Parquet: 67.20 KB


In [0]:
%python
#operacion acid basicas sobre delta lake

print("Historial inicial de la tabla Delta:")
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

Historial inicial de la tabla Delta:


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-29T16:08:08.000Z,71958481127059,daniicadenam@gmail.com,MERGE,"Map(predicate -> [""((coin#13085 = coin#13074) AND (date#13088 = date#13077))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3905647433407170),d630d470-c7c8-49f8-97df-ebd670db7c58,0829-155941-64ny18i6-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3576, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 18, executionTimeMs -> 6944, materializeSourceTimeMs -> 228, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1993, numTargetRowsUpdated -> 18, numOutputRows -> 19, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4628)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
4,2026-08-29T16:08:00.000Z,71958481127059,daniicadenam@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3905647433407170),9cb72036-2004-4c3b-b77b-dda3fa404389,0829-155941-64ny18i6-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 47808, p25FileSize -> 44069, numDeletionVectorsRemoved -> 2, conflictDetectionTimeMs -> 98, minFileSize -> 44069, numAddedFiles -> 1, maxFileSize -> 44069, p75FileSize -> 44069, p50FileSize -> 44069, numAddedBytes -> 44069)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
3,2026-08-29T16:07:59.000Z,71958481127059,daniicadenam@gmail.com,DELETE,"Map(predicate -> [""(price#12865 < 0.0)""])",null,List(3905647433407170),77f78f8e-d70b-4603-a073-95b598adc9f1,0829-155941-64ny18i6-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 457, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 452, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-29T16:07:57.000Z,71958481127059,daniicadenam@gmail.com,UPDATE,"Map(predicate -> [""(coin#12431 = bitcoin)""])",null,List(3905647433407170),9cb72036-2004-4c3b-b77b-dda3fa404389,0829-155941-64ny18i6-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 8255, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4555, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2070, numAddedFiles -> 1, numUpdatedRows -> 1442, numAddedBytes -> 12595, rewriteTimeMs -> 2456)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
1,2026-08-29T16:07:49.000Z,71958481127059,daniicadenam@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3905647433407170),f3930325-c84e-4214-8eda-ea38115773ac,0829-155941-64ny18i6-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1747)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-29T16:07:37.000Z,71958481127059,daniicadenam@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> 

In [0]:
%python
#insert

# INSERT / APPEND
 
new_rows = [
    Row(
        coin="bitcoin",
        vs_currency="usd",
        timestamp=datetime.now(UTC),
        date=date.today(),
        price=99999.99
    ),
    Row(
        coin="ethereum",
        vs_currency="usd",
        timestamp=datetime.now(UTC),
        date=date.today(),
        price=4999.99
    )
]
 
new_df = spark.createDataFrame(new_rows)
 
(
    new_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(FULL_TABLE_NAME)
)
 
print("Nuevos registros insertados.")

Nuevos registros insertados.


In [0]:
%python
#consultar nuevos registros
display(
    spark.sql(f"""
    SELECT *
    FROM {FULL_TABLE_NAME}
    ORDER BY timestamp DESC
    LIMIT 10
    """)
)

coin,vs_currency,timestamp,date,price
ethereum,usd,2026-08-29T16:07:46.704Z,2026-08-29,4999.99
bitcoin,usd,2026-08-29T16:07:46.704Z,2026-08-29,99999.99
solana,usd,2026-08-29T16:03:50.000Z,2026-08-29,105.13493385347978
ethereum,usd,2026-08-29T16:03:30.000Z,2026-08-29,2446.1678560309147
bitcoin,usd,2026-08-29T16:02:40.000Z,2026-08-29,77883.78572375458
bitcoin,usd,2026-08-29T15:00:00.000Z,2026-08-29,78010.76127895089
ethereum,usd,2026-08-29T15:00:00.000Z,2026-08-29,2448.444625275602
solana,usd,2026-08-29T15:00:00.000Z,2026-08-29,105.36209114081686
bitcoin,usd,2026-08-29T14:00:00.000Z,2026-08-29,77699.90411195564
ethereum,usd,2026-08-29T14:00:00.000Z,2026-08-29,2435.70991057065


In [0]:
%python
#update

# UPDATE
 
spark.sql(f"""
UPDATE {FULL_TABLE_NAME}
SET price = price * 1.01
WHERE coin = 'bitcoin'
""")
 
print("UPDATE ejecutado sobre registros de solana.")



UPDATE ejecutado sobre registros de solana.


In [0]:
%python

# DELETE
 
spark.sql(f"""
DELETE FROM {FULL_TABLE_NAME}
WHERE price < 0
""")
 
print("DELETE ejecutado.")

DELETE ejecutado.


In [0]:
%python

# MERGE
 
updates_data = [
    ("bitcoin", "usd", datetime.now(UTC), date.today(), 88888.88),
    ("cardano", "usd", datetime.now(UTC), date.today(), 0.75)
]
 
updates_pd = pd.DataFrame(
    updates_data,
    columns=["coin", "vs_currency", "timestamp", "date", "price"]
)
 
updates_spark = spark.createDataFrame(updates_pd)
updates_spark.createOrReplaceTempView("coin_updates")
 
spark.sql(f"""
MERGE INTO {FULL_TABLE_NAME} AS target
USING coin_updates AS source
ON target.coin = source.coin AND target.date = source.date
WHEN MATCHED THEN
  UPDATE SET
    target.vs_currency = source.vs_currency,
    target.timestamp = source.timestamp,
    target.price = source.price
WHEN NOT MATCHED THEN
  INSERT (coin, vs_currency, timestamp, date, price)
  VALUES (source.coin, source.vs_currency, source.timestamp, source.date, source.price)
""")
 
print("MERGE ejecutado.")

MERGE ejecutado.


In [0]:
%python
#consultar conteos por versiones
spark.read.format("delta").option("versionAsOf", 5).table(FULL_TABLE_NAME).count()

4325

In [0]:
%python


display(
    spark.sql(f"""
    SELECT *
    FROM {FULL_TABLE_NAME}
    ORDER BY timestamp DESC
    LIMIT 10
    """)
)

coin,vs_currency,timestamp,date,price
cardano,usd,2026-08-29T16:07:58.711Z,2026-08-29,0.75
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
bitcoin,usd,2026-08-29T16:07:58.711Z,2026-08-29,88888.88
